In [1]:
!git clone https://github.com/yuji4/MedSeg3D-KO.git

Cloning into 'MedSeg3D-KO'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 44 (delta 5), reused 39 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 31.71 KiB | 5.28 MiB/s, done.
Resolving deltas: 100% (5/5), done.


In [6]:
!pip install -r /content/MedSeg3D-KO/requirements.txt

  Using cached torch-2.2.1-cp312-cp312-manylinux1_x86_64.whl.metadata (26 kB)
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylin

In [1]:
!git clone https://github.com/BAAI-DCAI/M3D.git

Cloning into 'M3D'...
remote: Enumerating objects: 305, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 305 (delta 90), reused 77 (delta 77), pack-reused 178 (from 1)
Receiving objects: 100% (305/305), 37.16 MiB | 23.17 MiB/s, done.
Resolving deltas: 100% (121/121), done.


In [1]:
# 셀 3: 모델 로드
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "GoodBaiBai88/M3D-LaMed-Phi-3-4B"
dtype = torch.bfloat16

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    model_max_length=512,
    padding_side="right",
    use_fast=False,
    trust_remote_code=True
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=dtype,
    device_map='auto',
    trust_remote_code=True
)
model.eval()
print("로드 성공")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
/usr/local/lib/python3.12/dist-packages/monai/utils/deprecate_utils.py:221: FutureWarning: monai.networks.blocks.patchembedding PatchEmbeddingBlock.__init__:pos_embed: Argument `pos_embed` has been deprecated since version 1.2. It will be removed in version 1.4. please use `proj_type` instead.
  warn_deprecated(argname, msg, warning_category)
/usr/local/lib/pyth

build_sam_vit_3d...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

로드 성공


In [3]:
%cd MedSeg3D-KO

/content/MedSeg3D-KO


In [4]:
import sys
sys.path.insert(0, '/content/MedSeg3D-KO')

from src.inference.segmentation import SegmentationPipeline

pipeline = SegmentationPipeline()
pipeline.model = model
pipeline.tokenizer = tokenizer
pipeline._device = next(model.parameters()).device
print("pipeline 준비 완료")

pipeline 준비 완료


데이터 다운로드

In [24]:
!wget "https://assets.ngc.nvidia.com/products/api-catalog/vista3d/example-1.nii.gz" -O /content/CT_Abdo.nii.gz

--2026-05-21 06:29:39--  https://assets.ngc.nvidia.com/products/api-catalog/vista3d/example-1.nii.gz
Resolving assets.ngc.nvidia.com (assets.ngc.nvidia.com)... 18.65.3.3, 18.65.3.50, 18.65.3.90, ...
Connecting to assets.ngc.nvidia.com (assets.ngc.nvidia.com)|18.65.3.3|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 33037057 (32M) [binary/octet-stream]
Saving to: ‘/content/CT_Abdo.nii.gz’

/content/CT_Abdo.ni 100%[===================>]  31.51M  91.1MB/s    in 0.3s    

2026-05-21 06:29:39 (91.1 MB/s) - ‘/content/CT_Abdo.nii.gz’ saved [33037057/33037057]



In [27]:
!pip install TotalSegmentator
!wget https://zenodo.org/record/6802614/files/Totalsegmentator_dataset_small_v201.zip -O /content/total_small.zip

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.6/205.6 kB 9.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.7/100.7 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

--2026-05-21 06:37:56--  https://zenodo.org/record/6802614/files/Totalsegmentator_dataset_small_v201.zip
Resolving zenodo.org (zenodo.org)... 188.185.43.153, 188.185.48.75, 188.184.98.114, ...
Connecting to zenodo.org (zenodo.org)|188.185.43.153|:443... connected.
HTTP request sent, awaiting response... 301 MOVED PERMANENTLY
Location: /records/6802614/files/Totalsegmentator_dataset_small_v201.zip [following]
--2026-05-21 06:37:57--  https://zenodo.org/records/6802614/files/Totalsegmentator_dataset_small_v201.zip
Reusing existing connection to zenodo.org:443.
HTTP request sent, awaiting response... 404 NOT FOUND
2026-05-21 06:37:57 ERROR 404: NOT FOUND.



In [2]:
!wget "https://assets.ngc.nvidia.com/products/api-catalog/vista3d/example-2.nii.gz" -O /content/CT_2.nii.gz
!wget "https://assets.ngc.nvidia.com/products/api-catalog/vista3d/example-3.nii.gz" -O /content/CT_3.nii.gz

--2026-05-21 06:40:26--  https://assets.ngc.nvidia.com/products/api-catalog/vista3d/example-2.nii.gz
Resolving assets.ngc.nvidia.com (assets.ngc.nvidia.com)... 18.155.192.104, 18.155.192.84, 18.155.192.42, ...
Connecting to assets.ngc.nvidia.com (assets.ngc.nvidia.com)|18.155.192.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 14721727 (14M) [binary/octet-stream]
Saving to: ‘/content/CT_2.nii.gz’

/content/CT_2.nii.g 100%[===================>]  14.04M  71.9MB/s    in 0.2s    

2026-05-21 06:40:27 (71.9 MB/s) - ‘/content/CT_2.nii.gz’ saved [14721727/14721727]

--2026-05-21 06:40:27--  https://assets.ngc.nvidia.com/products/api-catalog/vista3d/example-3.nii.gz
Resolving assets.ngc.nvidia.com (assets.ngc.nvidia.com)... 18.155.192.104, 18.155.192.84, 18.155.192.42, ...
Connecting to assets.ngc.nvidia.com (assets.ngc.nvidia.com)|18.155.192.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 32788698 (31M) [binary/octet-stream]
Saving t

In [3]:
import nibabel as nib
for i in [2, 3]:
    nii = nib.load(f'/content/CT_{i}.nii.gz')
    print(f"CT_{i} shape:", nii.shape, "spacing:", nii.header.get_zooms())

CT_2 shape: (235, 235, 203) spacing: (np.float32(1.5), np.float32(1.5), np.float32(1.5))
CT_3 shape: (248, 248, 437) spacing: (np.float32(1.5), np.float32(1.5), np.float32(1.5))


app 실행 예시

In [13]:
import sys

for key in list(sys.modules.keys()):
    if 'src' in key or 'app' in key:
        del sys.modules[key]

sys.path.insert(0, '/content/MedSeg3D-KO')

from src.inference.segmentation import SegmentationPipeline

pipeline2 = SegmentationPipeline()
pipeline2.model = model
pipeline2.tokenizer = tokenizer
pipeline2._device = next(model.parameters()).device

import app.gradio_app as app_module
app_module._pipeline = pipeline2
app_module.demo.queue()
app_module.demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a6eb51809a60776c4c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
